# Sana-0.6B BSS/BDS Colab runner

This notebook follows the same path and git-pull style as `model_a_relitlive_official_bss_run_all.ipynb`: clone/pull code into `/content`, keep run outputs under a local run root, mirror heavy artifacts to Drive, and only run the mini-suite after smoke passes.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/WANG-Ruipeng/Sana.git"
BRANCH = "sana06b-bss-bds"
REPO_DIR = Path("/content/Sana-BSS")
SANA_UPSTREAM_URL = "https://github.com/NVlabs/Sana.git"
SANA_UPSTREAM_DIR = Path("/content/Sana")

RUN_NAME = "sana06b_bss_bds_v1"
RUNS_ROOT = Path("/content/Sana-BSS-Runs")
DRIVE_RUNS_ROOT = Path("/content/drive/MyDrive/Colab_Projects/Sana-BSS-BDS")
RUN_ROOT = RUNS_ROOT / RUN_NAME
DRIVE_RUN_ROOT = DRIVE_RUNS_ROOT / RUN_NAME

# Direct full run requested: skip smoke and run the 16-prompt mini-suite.
RUN_SMOKE = False
RUN_MINI_SUITE = True

# Install repo/minimal dependencies before smoke/full inference.
INSTALL_DEPS = True
RUN_ENV_SETUP = False

# Set True only if you want to discard the existing /content/Sana-BSS checkout.
FORCE_RECLONE = False
MOUNT_DRIVE = True

# Large weights are not stored in git.
DRIVE_WEIGHTS_ROOT = Path("/content/drive/MyDrive/ModelWeights/Sana")
WEIGHTS_SOURCE_DIR = DRIVE_WEIGHTS_ROOT / "Sana_600M_1024px"
WEIGHTS_RUN_DIR = WEIGHTS_SOURCE_DIR

# CPU-safe download mode. Set both True to download weights to Drive and stop before inference.
DOWNLOAD_WEIGHTS_ONLY = False
AUTO_DOWNLOAD_FROM_HF = False
HF_MODEL_ID = "Efficient-Large-Model/Sana_600M_1024px"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

BACKEND = "native"
CONFIG_PATH = "configs/sana_config/1024ms/Sana_600M_img1024.yaml"
SCRIPT_ROOT = REPO_DIR / "bss_experiments/sana06b_bss_bds_v1/scripts"

print("REPO_URL:", REPO_URL)
print("BRANCH:", BRANCH)
print("REPO_DIR:", REPO_DIR)
print("RUN_ROOT:", RUN_ROOT)
print("DRIVE_RUN_ROOT:", DRIVE_RUN_ROOT)
print("RUN_SMOKE:", RUN_SMOKE)
print("RUN_MINI_SUITE:", RUN_MINI_SUITE)
print("WEIGHTS_SOURCE_DIR:", WEIGHTS_SOURCE_DIR)


In [ ]:
import getpass
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped or failed:", repr(exc))

RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Runtime setup done.")


In [ ]:
def _display_cmd(cmd):
    parts = [str(part) for part in cmd]
    masked = []
    skip_next = False
    for part in parts:
        if skip_next:
            masked.append("***")
            skip_next = False
            continue
        masked.append(part)
        if part == "--token":
            skip_next = True
    text = " ".join(masked)
    if "x-access-token:" in text:
        text = text.split("x-access-token:")[0] + "x-access-token:***@" + text.split("@", 1)[-1]
    return text


def run(cmd, cwd=None, check=True):
    print("$", _display_cmd(cmd))
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True)


def run_checked(cmd, cwd=None, env=None, check=True):
    print("$", _display_cmd(cmd))
    proc = subprocess.run(
        [str(part) for part in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if proc.stdout:
        print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {_display_cmd(cmd)}")
    return proc


def authenticated_url(url):
    token = os.environ.get("GITHUB_TOKEN")
    if token is None:
        token = getpass.getpass("GitHub token for clone/fetch; leave blank if the branch is public: ")
        if token:
            os.environ["GITHUB_TOKEN"] = token
    if token:
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url


def huggingface_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token is None:
        token = getpass.getpass("Hugging Face token for model download; leave blank if the model is public: ")
        if token:
            os.environ["HF_TOKEN"] = token
    return token or ""


def checkout_repo(repo_url, branch, repo_dir, *, force_reclone=False, single_branch=True):
    clone_url = authenticated_url(repo_url)
    repo_dir = Path(repo_dir)

    if force_reclone and repo_dir.exists():
        shutil.rmtree(repo_dir)

    if repo_dir.exists() and not (repo_dir / ".git").exists():
        fallback = repo_dir.with_name(f"{repo_dir.name}_non_git_{int(time.time())}")
        print(f"Existing non-git directory found at {repo_dir}; moving it to {fallback}")
        shutil.move(str(repo_dir), str(fallback))

    if not repo_dir.exists():
        cmd = ["git", "clone"]
        if single_branch:
            cmd += ["--branch", branch, "--single-branch"]
        cmd += [clone_url, str(repo_dir)]
        run(cmd)
    else:
        run(["git", "-C", str(repo_dir), "remote", "set-url", "origin", clone_url])
        run(["git", "-C", str(repo_dir), "fetch", "origin", branch])
        run(["git", "-C", str(repo_dir), "switch", "-C", branch, f"origin/{branch}"])
        run(["git", "-C", str(repo_dir), "reset", "--hard", f"origin/{branch}"])

    # Avoid leaving a token-bearing URL in .git/config.
    run(["git", "-C", str(repo_dir), "remote", "set-url", "origin", repo_url])
    run(["git", "-C", str(repo_dir), "branch", "--show-current"])
    run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"])
    run(["git", "-C", str(repo_dir), "status", "--short"])


checkout_repo(REPO_URL, BRANCH, REPO_DIR, force_reclone=FORCE_RECLONE, single_branch=True)

# Keep an official upstream checkout for audit/comparison, but the experiment code runs from REPO_DIR.
if not SANA_UPSTREAM_DIR.exists():
    run(["git", "clone", SANA_UPSTREAM_URL, str(SANA_UPSTREAM_DIR)])
else:
    run(["git", "-C", str(SANA_UPSTREAM_DIR), "fetch", "origin", "main"], check=False)
    run(["git", "-C", str(SANA_UPSTREAM_DIR), "switch", "main"], check=False)
    run(["git", "-C", str(SANA_UPSTREAM_DIR), "reset", "--hard", "origin/main"], check=False)
run(["git", "-C", str(SANA_UPSTREAM_DIR), "rev-parse", "HEAD"], check=False)


In [ ]:
os.environ["PYTHONPATH"] = f"{REPO_DIR}:{SANA_UPSTREAM_DIR}:" + os.environ.get("PYTHONPATH", "")

if DOWNLOAD_WEIGHTS_ONLY and not INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    print("Installed huggingface_hub for download-only mode.")
elif INSTALL_DEPS:
    if RUN_ENV_SETUP:
        run(["bash", "environment_setup.sh", "sana"], cwd=REPO_DIR)
    else:
        run([
            sys.executable, "-m", "pip", "install", "-q",
            "diffusers>=0.32.0", "transformers", "accelerate", "safetensors",
            "sentencepiece", "huggingface_hub", "opencv-python", "imageio",
            "pandas", "numpy", "pillow", "matplotlib", "omegaconf", "pyrallis", "termcolor",
            "einops", "ftfy", "timm==0.6.13", "flash-linear-attention>=0.4.2", "liger-kernel", "qwen-vl-utils"
        ])
    import importlib.util
    required_modules = {
        "fla": "flash-linear-attention>=0.4.2",
        "einops": "einops",
        "timm": "timm==0.6.13",
        "liger_kernel": "liger-kernel",
        "qwen_vl_utils": "qwen-vl-utils",
        "pyrallis": "pyrallis",
        "omegaconf": "omegaconf",
        "termcolor": "termcolor",
    }
    missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
    if missing_modules:
        packages = [required_modules[name] for name in missing_modules]
        print("Installing missing native Sana modules:", packages)
        run([sys.executable, "-m", "pip", "install", "-q", *packages])
        still_missing = [name for name in missing_modules if importlib.util.find_spec(name) is None]
        if still_missing:
            raise ModuleNotFoundError(f"Missing Python modules after dependency install: {still_missing}")
    print("Dependency install/check completed.")
else:
    print("INSTALL_DEPS=False; skipping dependency installation.")


In [ ]:
DOWNLOAD_CELL_VERSION = "snapshot_download_v2_no_cli"
print("DOWNLOAD_CELL_VERSION:", DOWNLOAD_CELL_VERSION)

from huggingface_hub import snapshot_download, whoami


def sana_weights_valid(path):
    path = Path(path)
    candidates = [
        path / "checkpoints/Sana_600M_1024px.pth",
        path / "checkpoint/Sana_600M_1024px.pth",
        path / "Sana_600M_1024px.pth",
    ]
    return path.is_dir() and (any(p.exists() for p in candidates) or any(path.rglob("*.pth")))

print("WEIGHTS_SOURCE_DIR:", WEIGHTS_SOURCE_DIR)
print("weights valid:", sana_weights_valid(WEIGHTS_SOURCE_DIR))
if WEIGHTS_SOURCE_DIR.exists():
    print("pth files:", [str(p) for p in WEIGHTS_SOURCE_DIR.rglob("*.pth")][:8])

if AUTO_DOWNLOAD_FROM_HF and not sana_weights_valid(WEIGHTS_SOURCE_DIR):
    WEIGHTS_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    token = HF_TOKEN or huggingface_token()
    if token:
        try:
            info = whoami(token=token)
            print("HF auth OK:", info.get("name") or info.get("fullname") or "authenticated")
        except Exception as exc:
            print("HF whoami failed; download will still be attempted:", repr(exc))

    print("Downloading with huggingface_hub.snapshot_download; no HF token is passed on the command line.")
    snapshot_download(
        repo_id=HF_MODEL_ID,
        repo_type="model",
        local_dir=str(WEIGHTS_SOURCE_DIR),
        token=token or None,
        resume_download=True,
    )

if not sana_weights_valid(WEIGHTS_SOURCE_DIR):
    print("Missing Sana-0.6B weights. Put them on Drive or set AUTO_DOWNLOAD_FROM_HF=True.")
    print(f"Expected: {WEIGHTS_SOURCE_DIR}")
    print("Manual Colab fallback:")
    print(f"from huggingface_hub import snapshot_download")
    print(f"snapshot_download(repo_id='{HF_MODEL_ID}', repo_type='model', local_dir='{WEIGHTS_SOURCE_DIR}', token='YOUR_TOKEN_OR_NONE')")
    raise FileNotFoundError("Missing Sana-0.6B weights")

print("Sana-0.6B weights are ready on Drive:", WEIGHTS_SOURCE_DIR)
if DOWNLOAD_WEIGHTS_ONLY:
    print("DOWNLOAD_WEIGHTS_ONLY=True; stopping before preflight/inference. You can switch to GPU later for smoke.")
    raise SystemExit("Download-only completed")


In [ ]:
required_paths = {
    "experiment scripts": SCRIPT_ROOT,
    "adapter": SCRIPT_ROOT / "sana06b_adapter.py",
    "native pipeline": REPO_DIR / "app/sana_pipeline.py",
    "600M config": REPO_DIR / CONFIG_PATH,
    "weights dir": WEIGHTS_RUN_DIR,
}

missing = []
for label, path in required_paths.items():
    ok = sana_weights_valid(path) if label == "weights dir" else Path(path).exists()
    print(f"{label:24s} {'OK' if ok else 'MISSING'}  {path}")
    if not ok:
        missing.append((label, path))

if missing:
    for label, path in missing:
        print(f"- {label}: {path}")
    raise FileNotFoundError("Missing required Sana BSS/BDS paths")

os.environ.update({
    "DRIVE_WEIGHTS_ROOT": str(DRIVE_WEIGHTS_ROOT),
    "DRIVE_EXPERIMENT_ROOT": str(DRIVE_RUN_ROOT),
    "SANA06B_WEIGHTS_DIR": str(WEIGHTS_RUN_DIR),
})

print("Preflight paths look ready.")


In [ ]:
# Colab hotfix: native Sana imports a small mmcv surface, but full mmcv 1.7.x is brittle on Colab Python 3.12.
# Create a package shim instead of a single mmcv.py file; Sana imports mmcv.utils and mmcv.runner.
import importlib
import shutil
import sys

bad_mmcv_file = REPO_DIR / "mmcv.py"
if bad_mmcv_file.exists():
    bad_mmcv_file.unlink()
    print("Removed old invalid mmcv.py shim:", bad_mmcv_file)
pycache_dir = REPO_DIR / "__pycache__"
if pycache_dir.exists():
    for cached in pycache_dir.glob("mmcv*.pyc"):
        cached.unlink()

mmcv_pkg = REPO_DIR / "mmcv"
if mmcv_pkg.exists():
    shutil.rmtree(mmcv_pkg)
mmcv_utils_pkg = mmcv_pkg / "utils"
mmcv_runner_pkg = mmcv_pkg / "runner"
mmcv_utils_pkg.mkdir(parents=True, exist_ok=True)
mmcv_runner_pkg.mkdir(parents=True, exist_ok=True)

(mmcv_pkg / "__init__.py").write_text(r'''
import os
import pickle
from pathlib import Path


class Config(dict):
    def __init__(self, cfg_dict=None, **kwargs):
        super().__init__()
        data = {}
        if cfg_dict:
            data.update(dict(cfg_dict))
        data.update(kwargs)
        for key, value in data.items():
            self[key] = self._wrap(value)

    @staticmethod
    def _wrap(value):
        if isinstance(value, dict) and not isinstance(value, Config):
            return Config(value)
        if isinstance(value, list):
            return [Config._wrap(item) for item in value]
        return value

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError as exc:
            raise AttributeError(name) from exc

    def __setattr__(self, name, value):
        self[name] = self._wrap(value)

    def merge_from_dict(self, options):
        for key, value in dict(options).items():
            self[key] = self._wrap(value)

    @classmethod
    def fromfile(cls, filename):
        import yaml
        with open(filename, encoding="utf-8") as handle:
            data = yaml.safe_load(handle) or {}
        return cls(data)


def mkdir_or_exist(dir_name):
    Path(dir_name).mkdir(parents=True, exist_ok=True)


def dump(obj, file):
    with open(file, "wb") as handle:
        pickle.dump(obj, handle)


def load(file):
    with open(file, "rb") as handle:
        return pickle.load(handle)


class Registry:
    def __init__(self, name, *args, **kwargs):
        self.name = name
        self.module_dict = {}
        self._module_dict = self.module_dict

    def __contains__(self, key):
        return key in self.module_dict

    def get(self, key):
        return self.module_dict.get(key)

    def register_module(self, module=None, name=None, force=False):
        def _register(cls):
            module_name = name or cls.__name__
            if not force and module_name in self.module_dict:
                raise KeyError(f"{module_name} is already registered in {self.name}")
            self.module_dict[module_name] = cls
            return cls
        if module is not None:
            return _register(module)
        return _register

    def build(self, cfg, default_args=None):
        return build_from_cfg(cfg, self, default_args=default_args)


def build_from_cfg(cfg, registry, default_args=None):
    if cfg is None:
        raise TypeError("cfg must not be None")
    if isinstance(cfg, str):
        cfg = {"type": cfg}
    elif hasattr(cfg, "to_dict"):
        cfg = cfg.to_dict()
    else:
        cfg = dict(cfg)
    args = dict(default_args or {})
    args.update({k: v for k, v in cfg.items() if k != "type"})
    obj_type = cfg.get("type")
    if isinstance(obj_type, str):
        obj_cls = registry.get(obj_type)
        if obj_cls is None:
            raise KeyError(f"{obj_type} is not registered in {registry.name}")
    else:
        obj_cls = obj_type
    return obj_cls(**args)
'''.lstrip(), encoding="utf-8")

(mmcv_utils_pkg / "__init__.py").write_text(r'''
try:
    from torch.nn.modules.batchnorm import _BatchNorm
    from torch.nn.modules.instancenorm import _InstanceNorm
except Exception:
    _BatchNorm = tuple()
    _InstanceNorm = tuple()
'''.lstrip(), encoding="utf-8")
(mmcv_utils_pkg / "logging.py").write_text("logger_initialized = {}\n", encoding="utf-8")
(mmcv_runner_pkg / "__init__.py").write_text(r'''
import torch
from mmcv import Registry

OPTIMIZERS = Registry("optimizer")
OPTIMIZER_BUILDERS = Registry("optimizer builder")


class DefaultOptimizerConstructor:
    def __init__(self, optimizer_cfg, paramwise_cfg=None):
        self.optimizer_cfg = dict(optimizer_cfg or {})
        self.paramwise_cfg = paramwise_cfg or {}
        self.base_lr = self.optimizer_cfg.get("lr")
        self.base_wd = self.optimizer_cfg.get("weight_decay")

    def __call__(self, model):
        return build_optimizer(model, self.optimizer_cfg)

    def add_params(self, params, module, prefix="", is_dcn_module=None):
        params.extend({"params": [param], "name": name} for name, param in module.named_parameters(recurse=False))

    @staticmethod
    def _is_in(param_group, params):
        target = set(param_group.get("params", []))
        return any(bool(target.intersection(set(group.get("params", [])))) for group in params)


def get_dist_info():
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        return torch.distributed.get_rank(), torch.distributed.get_world_size()
    return 0, 1


def build_optimizer(model, optimizer_cfg):
    cfg = dict(optimizer_cfg or {})
    opt_type = cfg.pop("type", "AdamW")
    if isinstance(opt_type, str):
        opt_cls = getattr(torch.optim, opt_type)
    else:
        opt_cls = opt_type
    return opt_cls(model.parameters(), **cfg)
'''.lstrip(), encoding="utf-8")
print("Wrote Colab mmcv package shim:", mmcv_pkg)

# Verify the current notebook process and child subprocesses will resolve the package shim.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
for module_name in ["mmcv", "mmcv.utils", "mmcv.utils.logging", "mmcv.runner"]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
from mmcv import Config as _ColabMMCVConfig
from mmcv import Registry as _ColabMMCVRegistry
from mmcv.runner import get_dist_info as _colab_get_dist_info
from mmcv.utils.logging import logger_initialized as _colab_mmcv_logger_initialized
print("Colab mmcv shim import OK:", _ColabMMCVRegistry, _ColabMMCVConfig, _colab_get_dist_info())

for subdir in ["reports", "tables", "metrics", "figures", "schedules", "manifests", "logs", "runtimes", "images"]:
    (RUN_ROOT / subdir).mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)

run([sys.executable, str(SCRIPT_ROOT / "make_manifest_sana06b_bds.py"), "--experiment-root", str(RUN_ROOT), "--backend", BACKEND], cwd=REPO_DIR)
run([sys.executable, str(SCRIPT_ROOT / "audit_sana06b.py"), "--weights-dir", str(WEIGHTS_RUN_DIR), "--output", str(RUN_ROOT / "reports/00_repo_model_hardware_audit.md")], cwd=REPO_DIR)
run([sys.executable, str(SCRIPT_ROOT / "validate_schedules.py"), "--output", str(RUN_ROOT / "schedules/schedule_validation.json")], cwd=REPO_DIR)


In [ ]:
def run_stream(cmd, cwd=REPO_DIR):
    print("$", _display_cmd(cmd))
    env = os.environ.copy()
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {_display_cmd(cmd)}")

if RUN_SMOKE:
    run_stream([
        sys.executable, str(SCRIPT_ROOT / "run_manifest.py"),
        "--manifest", str(RUN_ROOT / "manifests/sana06b_smoke_manifest.csv"),
        "--experiment-root", str(RUN_ROOT),
        "--backend", BACKEND,
        "--weights-dir", str(WEIGHTS_RUN_DIR),
        "--config-path", CONFIG_PATH,
        "--resume",
        "--sync_drive",
        "--drive-experiment-root", str(DRIVE_RUN_ROOT),
    ])
    run([sys.executable, str(SCRIPT_ROOT / "compute_metrics_against_ref.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_smoke_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "generate_figures.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_smoke_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
else:
    print("RUN_SMOKE=False; smoke was not started.")


In [ ]:
if RUN_MINI_SUITE:
    run_stream([
        sys.executable, str(SCRIPT_ROOT / "run_manifest.py"),
        "--manifest", str(RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv"),
        "--experiment-root", str(RUN_ROOT),
        "--backend", BACKEND,
        "--weights-dir", str(WEIGHTS_RUN_DIR),
        "--config-path", CONFIG_PATH,
        "--resume",
        "--sync_drive",
        "--drive-experiment-root", str(DRIVE_RUN_ROOT),
    ])
    run([sys.executable, str(SCRIPT_ROOT / "compute_metrics_against_ref.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "compute_bds.py"), "--metrics-dir", str(RUN_ROOT / "metrics")], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "generate_figures.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "write_final_report.py"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
else:
    print("RUN_MINI_SUITE=False, so the 16-prompt mini-suite was not started.")


In [ ]:
important_paths = {
    "experiment folder": RUN_ROOT,
    "Drive mirror": DRIVE_RUN_ROOT,
    "audit report": RUN_ROOT / "reports/00_repo_model_hardware_audit.md",
    "smoke manifest": RUN_ROOT / "manifests/sana06b_smoke_manifest.csv",
    "mini manifest": RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv",
    "metrics CSV": RUN_ROOT / "metrics/master_long_metrics.csv",
    "BDS table": RUN_ROOT / "tables/tableA_sana06b_bds_by_split.md",
    "paper row": RUN_ROOT / "tables/table_cross_model_same_compute_sana06b_row.md",
    "LaTeX row": RUN_ROOT / "tables/table_cross_model_same_compute_sana06b_row.tex",
    "final report": RUN_ROOT / "reports/FINAL_SANA06B_BSS_BDS_REPORT.md",
    "side-by-side index": RUN_ROOT / "figures/side_by_side/index.html",
}

for label, path in important_paths.items():
    print(f"{label:24s} {'OK' if path.exists() else 'not yet'}  {path}")

paper_row = RUN_ROOT / "tables/table_cross_model_same_compute_sana06b_row.md"
if paper_row.exists():
    print("\n===== Paper-format Row =====")
    print(paper_row.read_text(encoding="utf-8"))

final_report = RUN_ROOT / "reports/FINAL_SANA06B_BSS_BDS_REPORT.md"
if final_report.exists():
    print("\n===== Final Report Path =====")
    print(final_report)
